# Cohort Feature Engineering

- `vehicle_lake_raw.csv` 백업본을 불러온다.
- 공백 문자열을 `None`으로 치환한다.
- cohort grouping에 필요한 핵심 feature group을 먼저 정의한다.
- `year_label`은 core feature가 아니라 filtering / QA 용 metadata로만 본다.
- 전체 컬럼과 core feature의 null 비율을 확인한다.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

CANDIDATE_PATHS = [
    Path('data.bak/vehicle_lake_raw.csv'),
    Path('raw/models_cohort/cohort_generation/data.bak/vehicle_lake_raw.csv'),
]

for candidate in CANDIDATE_PATHS:
    if candidate.exists():
        DATA_PATH = candidate.resolve()
        break
else:
    raise FileNotFoundError('vehicle_lake_raw.csv 백업본을 찾을 수 없습니다.')

DATA_PATH


In [ ]:
df_raw = pd.read_csv(DATA_PATH, dtype=str)
df_raw.shape


In [ ]:
pd.DataFrame({'column_name': df_raw.columns})


In [ ]:
df = df_raw.apply(
    lambda col: col.map(
        lambda value: None if isinstance(value, str) and value.strip() == '' else value
    )
)
df.shape


## Core Feature Groups

### 1. Identity
- `brand`
- `model_name`

이 그룹은 사람이 모델별로 큰 급과 차종을 수동으로 정리할 때 가장 먼저 쓰는 기준이다.

### 2. Normalization Input
- `level_name`
- `class_name`

이 그룹은 같은 모델 안에서 세부 파워트레인/트림 이름을 정규화할 때 사용한다.
예: `더 뉴쏘렌토(MQ4) 2.5 가솔린 2WD 시그니처` 같은 이름을 canonical naming으로 맞출 때 필요하다.

### 3. Linearity Base
- `launch_price_krw`
- `displacement_cc`

이 그룹은 급수 + 체급 번호를 선형적으로 만들기 위한 기본 축이다.
`launch_price_krw`는 초기 시장 포지션을, `displacement_cc`는 같은 급 안에서의 세부 체급 차이를 설명한다.

### 4. Body Size Bundle
- `body_length_mm`
- `body_width_mm`
- `body_height_mm`
- `wheelbase_mm`

이 네 컬럼은 각각 따로 보기보다 `body_size` 묶음으로 같이 다루는 것이 자연스럽다.
이후 feature engineering 단계에서는 이 묶음을 기반으로 `body_size_index` 같은 파생 수치를 만들 수 있다.

### 5. Support / Metadata
- `year_label`
- `fuel_type`
- `drive_type`
- `seats_count`

`year_label`은 cohort core feature로 쓰지 않는다. 대신 `2010년 이전 모델 제거`, coverage 점검, QA 용도로만 유지한다.
`fuel_type`, `drive_type`, `seats_count`는 나중에 세부 분기나 보조 검증용으로 볼 수 있지만 현재 core plot에서는 제외한다.


In [ ]:
identity_columns = ['brand', 'model_name']
normalization_columns = ['level_name', 'class_name']
linearity_columns = ['launch_price_krw', 'displacement_cc']
body_size_columns = ['body_length_mm', 'body_width_mm', 'body_height_mm', 'wheelbase_mm']
support_columns = ['year_label', 'fuel_type', 'drive_type', 'seats_count']

core_feature_groups = {
    'identity': identity_columns,
    'normalization': normalization_columns,
    'linearity': linearity_columns,
    'body_size': body_size_columns,
}

core_columns = [
    column
    for group_columns in core_feature_groups.values()
    for column in group_columns
]

support_feature_groups = {'support': support_columns}
{'core_feature_groups': core_feature_groups, 'support_feature_groups': support_feature_groups}


In [ ]:
null_summary = (
    pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'null_count': df.isna().sum(),
        'null_ratio': df.isna().mean(),
    })
    .assign(
        non_null_count=lambda frame: len(df) - frame['null_count'],
        null_pct=lambda frame: (frame['null_ratio'] * 100).round(2),
    )
    .sort_values(['null_ratio', 'null_count'], ascending=[False, False])
)

display(null_summary)

plot_frame = null_summary.sort_values('null_ratio', ascending=True)
fig, ax = plt.subplots(figsize=(12, 16))
ax.barh(plot_frame.index, plot_frame['null_pct'], color='steelblue')
ax.set_title('Null Ratio by Column')
ax.set_xlabel('Null Percentage (%)')
ax.set_ylabel('Column')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
for group_name, columns in core_feature_groups.items():
    display(Markdown(f'### {group_name}'))
    display(null_summary.loc[columns])


## Core Columns Null Check

아래 플롯은 전체 컬럼이 아니라, 위에서 선정한 core columns만 대상으로 결측 비율을 다시 본다.
x축은 결측 비율 퍼센테이지, y축은 feature 이름이다.


In [ ]:
core_null_summary = null_summary.loc[core_columns].sort_values('null_ratio', ascending=True)

display(core_null_summary)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(core_null_summary.index, core_null_summary['null_pct'], color='darkorange')
ax.set_title('Null Ratio for Core Columns')
ax.set_xlabel('Null Percentage (%)')
ax.set_ylabel('Core Feature')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## `year_label` Available Distribution by Brand / Model

`year_label`이 실제로 존재하는 row만 기준으로 `brand + model_name` 분포를 본다.
이 분포를 보면 연식 정보가 상대적으로 잘 채워져 있는 모델군이 무엇인지 바로 확인할 수 있다.


In [ ]:
year_label_available_distribution = (
    df[df['year_label'].notna()]
    .groupby(['brand', 'model_name'], dropna=False)
    .agg(
        row_count=('year_label', 'size'),
        unique_year_count=('year_label', lambda s: s.nunique(dropna=True)),
        sample_level_names=('level_name', lambda s: ', '.join(list(dict.fromkeys(str(v) for v in s.dropna()))[:3])),
    )
    .reset_index()
    .sort_values(['row_count', 'unique_year_count', 'brand', 'model_name'], ascending=[False, False, True, True])
)

year_label_available_distribution['brand_model'] = (
    year_label_available_distribution['brand'] + ' / ' + year_label_available_distribution['model_name']
)

display(year_label_available_distribution.head(50))

plot_frame = (
    year_label_available_distribution
    .head(40)
    .sort_values(['row_count', 'unique_year_count'], ascending=[True, True])
)

fig, ax = plt.subplots(figsize=(12, 14))
ax.barh(plot_frame['brand_model'], plot_frame['row_count'], color='seagreen')
ax.set_title('Rows with year_label by Brand / Model (Top 40)')
ax.set_xlabel('Row Count with year_label')
ax.set_ylabel('Brand / Model')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## Model Year Summary

`year_label`은 core cohort feature는 아니지만, filtering 용도로는 여전히 중요하다.
여기서는 `brand + model_name` 기준으로 사용 가능한 연식 범위를 요약해서 `2010년 이전 모델 제거` 후보를 본다.


In [ ]:
import re

def parse_year(value):
    if value is None:
        return pd.NA
    match = re.search(r'(19|20)\d{2}', str(value))
    if not match:
        return pd.NA
    return int(match.group(0))

df['year_value'] = df['year_label'].map(parse_year)

model_year_summary = (
    df.groupby(['brand', 'model_name'], dropna=False)
    .agg(
        total_rows=('model_name', 'size'),
        year_present_rows=('year_value', lambda s: s.notna().sum()),
        year_missing_rows=('year_value', lambda s: s.isna().sum()),
        min_year=('year_value', lambda s: int(s.dropna().min()) if s.notna().any() else pd.NA),
        max_year=('year_value', lambda s: int(s.dropna().max()) if s.notna().any() else pd.NA),
        unique_year_count=('year_value', lambda s: int(s.dropna().nunique()) if s.notna().any() else 0),
        sample_levels=('level_name', lambda s: ', '.join(list(dict.fromkeys(str(v) for v in s.dropna()))[:5])),
        sample_classes=('class_name', lambda s: ', '.join(list(dict.fromkeys(str(v) for v in s.dropna()))[:5])),
    )
    .reset_index()
)

model_year_summary['has_any_2010_or_earlier_year'] = model_year_summary['min_year'].map(lambda x: 'Y' if pd.notna(x) and int(x) <= 2010 else 'N')
model_year_summary['all_years_2010_or_earlier'] = model_year_summary['max_year'].map(lambda x: 'Y' if pd.notna(x) and int(x) <= 2010 else 'N')
model_year_summary['has_any_post_2010_year'] = model_year_summary['max_year'].map(lambda x: 'Y' if pd.notna(x) and int(x) > 2010 else 'N')
model_year_summary['all_year_missing'] = model_year_summary['year_present_rows'].map(lambda x: 'Y' if int(x) == 0 else 'N')
model_year_summary['drop_by_year_rule'] = model_year_summary.apply(
    lambda row: 'Y' if (row['all_years_2010_or_earlier'] == 'Y' or row['all_year_missing'] == 'Y') else 'N',
    axis=1,
)

model_year_summary = model_year_summary.sort_values(
    ['drop_by_year_rule', 'all_year_missing', 'all_years_2010_or_earlier', 'brand', 'model_name'],
    ascending=[False, False, False, True, True],
).reset_index(drop=True)

display(model_year_summary.head(50))


## Year-Based Drop Candidate Lists

- `has_any_2010_or_earlier_year == Y`: 연식 범위 안에 2010년 이하가 하나라도 포함되는 모델
- `all_years_2010_or_earlier == Y`: 확인된 모든 연식이 2010년 이하인 모델
- `all_year_missing == Y`: 연식 record가 아예 없는 모델
- `drop_by_year_rule == Y`: `2010년 이하만 있는 모델` 또는 `연식 record가 아예 없는 모델`


In [ ]:
models_with_any_2010_or_earlier_year = model_year_summary[model_year_summary['has_any_2010_or_earlier_year'] == 'Y'].copy()
models_only_2010_or_earlier = model_year_summary[model_year_summary['all_years_2010_or_earlier'] == 'Y'].copy()
models_with_all_year_missing = model_year_summary[model_year_summary['all_year_missing'] == 'Y'].copy()
models_to_drop_by_year_rule = model_year_summary[model_year_summary['drop_by_year_rule'] == 'Y'].copy()

print('models_with_any_2010_or_earlier_year:', len(models_with_any_2010_or_earlier_year))
print('models_only_2010_or_earlier:', len(models_only_2010_or_earlier))
print('models_with_all_year_missing:', len(models_with_all_year_missing))
print('models_to_drop_by_year_rule:', len(models_to_drop_by_year_rule))

display(models_to_drop_by_year_rule[['brand', 'model_name', 'min_year', 'max_year', 'total_rows', 'year_missing_rows', 'all_year_missing']].head(100))


## Remove Models with Only 2010-or-Earlier Years or No Year Records

`drop_by_year_rule == Y` 인 `brand + model_name` 조합을 전체 raw dataframe에서 제외한다.
즉, 확인 가능한 연식이 전부 2010년 이하이거나, 연식 record가 아예 없는 모델은 cohort grouping 대상에서 뺀다.


In [ ]:
models_to_drop_keys = set(
    zip(models_to_drop_by_year_rule['brand'], models_to_drop_by_year_rule['model_name'])
)

df_post_2010 = df.loc[
    ~df.apply(lambda row: (row['brand'], row['model_name']) in models_to_drop_keys, axis=1)
].copy()

print('original_rows:', len(df))
print('post_filter_rows:', len(df_post_2010))
print('removed_rows:', len(df) - len(df_post_2010))
print('removed_model_count:', len(models_to_drop_keys))

df_post_2010.shape


## `year_label` Available Distribution After Pre-2010 Removal

2010년 이전 모델을 제거한 뒤에도, `year_label`이 존재하는 `brand + model_name` 분포를 다시 확인한다.
이 플롯은 이후 수동 그룹화와 normalization 우선순위를 잡을 때 기준으로 사용한다.


In [ ]:
year_label_available_distribution_post_2010 = (
    df_post_2010[df_post_2010['year_label'].notna()]
    .groupby(['brand', 'model_name'], dropna=False)
    .agg(
        row_count=('year_label', 'size'),
        unique_year_count=('year_label', lambda s: s.nunique(dropna=True)),
        sample_level_names=('level_name', lambda s: ', '.join(list(dict.fromkeys(str(v) for v in s.dropna()))[:3])),
    )
    .reset_index()
    .sort_values(['row_count', 'unique_year_count', 'brand', 'model_name'], ascending=[False, False, True, True])
)

year_label_available_distribution_post_2010['brand_model'] = (
    year_label_available_distribution_post_2010['brand'] + ' / ' + year_label_available_distribution_post_2010['model_name']
)

display(year_label_available_distribution_post_2010.head(50))

    


## Null Ratio Recalculation for Core Columns After Pre-2010 Removal

`df_post_2010` 기준으로 다시 계산하되, 마지막 점검은 전체 컬럼이 아니라 core columns만 대상으로 본다.
즉, 2010년 이전 모델 제거 이후 핵심 feature availability가 어떻게 달라지는지 확인하는 셀이다.


In [ ]:
core_null_summary_post_2010 = (
    pd.DataFrame({
        'dtype': df_post_2010[core_columns].dtypes.astype(str),
        'null_count': df_post_2010[core_columns].isna().sum(),
        'null_ratio': df_post_2010[core_columns].isna().mean(),
    })
    .assign(
        non_null_count=lambda frame: len(df_post_2010) - frame['null_count'],
        null_pct=lambda frame: (frame['null_ratio'] * 100).round(2),
    )
    .sort_values(['null_ratio', 'null_count'], ascending=[False, False])
)

display(core_null_summary_post_2010)

plot_frame = core_null_summary_post_2010.sort_values('null_ratio', ascending=True)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plot_frame.index, plot_frame['null_pct'], color='mediumpurple')
ax.set_title('Core Column Null Ratio After Pre-2010 Removal')
ax.set_xlabel('Null Percentage (%)')
ax.set_ylabel('Core Feature')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## Export Final CSV Outputs

최종적으로는 `cohort_generation/data/` 아래에 실제로 사용할 최종 CSV만 저장한다.
현재 notebook 기준 최종 산출물은 아래 두 개만 남긴다.

- `cohort_grouping_input.csv`
- `model_inventory.csv`

`cohort_grouping_input.csv`에는 cohort grouping에 바로 쓸 핵심 컬럼만 남긴다.
`model_inventory.csv`는 `brand + model_name` 기준으로 한 줄씩 만들고, 모델명 정리와 이미지 정리용으로 출시 연도 범위와 예시 이름들을 같이 붙인다.


In [ ]:
OUTPUT_DIR = DATA_PATH.parent.parent / 'data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_columns = [
    'brand',
    'model_name',
    'level_name',
    'class_name',
    'launch_price_krw',
    'displacement_cc',
    'body_length_mm',
    'body_width_mm',
    'body_height_mm',
]

cohort_grouping_input = df_post_2010[export_columns].copy()

model_inventory = (
    df_post_2010.groupby(['brand', 'model_name'], dropna=False)
    .agg(
        first_model_year=('year_value', lambda s: int(s.dropna().min()) if s.notna().any() else pd.NA),
        last_model_year=('year_value', lambda s: int(s.dropna().max()) if s.notna().any() else pd.NA),
        submodel_line_examples=('level_name', lambda s: ', '.join(list(dict.fromkeys(str(v) for v in s.dropna()))[:10])),
        trim_name_examples=('class_name', lambda s: ', '.join(list(dict.fromkeys(str(v) for v in s.dropna()))[:10])),
    )
    .reset_index()
    .sort_values(['brand', 'model_name'])
)

cohort_grouping_input.to_csv(OUTPUT_DIR / 'cohort_grouping_input.csv', index=False)
model_inventory.to_csv(OUTPUT_DIR / 'model_inventory.csv', index=False)

pd.DataFrame({
    'output_file': [
        'cohort_grouping_input.csv',
        'model_inventory.csv',
    ],
    'row_count': [
        len(cohort_grouping_input),
        len(model_inventory),
    ],
})
